# Pertes d'un MOSFET seul

Calcul des pertes et de la température de jonction d'**un MOSFET unique**, à partir des
données constructeur du composant et de son driver.

**Mode d'emploi — 3 étapes :**

1. Exécuter la cellule de configuration (`Kernel → Restart & Run All` fait tout d'un coup).
2. Choisir le MOSFET, le driver et le point de fonctionnement dans le panneau.
3. Cliquer sur **Calculer**.

> Le recouvrement est un champ à remplir : le $Q_{rr}$ de la diode que l'amorçage
> force à se recouvrer — celle d'en face, pas celle de ce MOSFET. Laisser 0 s'il
> n'y en a pas.

In [1]:

# --- Configuration : à exécuter en premier -----------------------------------
import sys
from pathlib import Path

# Remonte jusqu'à la racine du projet (le dossier qui contient DATABASE/)
ROOT = Path.cwd()
while not (ROOT / "DATABASE").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import base64
import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

from DATABASE.db_driver_mosfet import DRIVER_LIBRARY, load_driver
from DATABASE.db_mosfet import MOSFET_LIBRARY, load_mosfet
from SRC.MOSFET.mosfet_loss import (
    OPERATING_POINT,
    loss_single_mosfet_at_temp,
    loss_thermal_iteration,
)
from SRC.MOSFET.mosfet_plot import loss_table, plot_loss_breakdown, plot_thermal_iteration
from SRC.THERMAL.dissipator import *

#%matplotlib inline

print(f"Racine du projet : {ROOT}")
print(f"MOSFET disponibles : {', '.join(sorted(MOSFET_LIBRARY))}")
print(f"Drivers disponibles : {', '.join(sorted(DRIVER_LIBRARY))}")

Racine du projet : c:\Users\tomro\Documents\MyCalculator_V3
MOSFET disponibles : BSC016N06NS
Drivers disponibles : UCC27714


---
## 1. Figure de référence — la charge de grille

Tous les temps de commutation du modèle sortent de ce diagramme. Les charges
$Q_{g(th)}$, $Q_{gs}$, $Q_{gd}$ et le plateau Miller sont ce qui pilote la vitesse
de commutation, donc les pertes.

Choisis l'image à afficher dans la liste (tout ce qui est dans `DOCUMENTS/`).

In [2]:
# --- Image de référence ------------------------------------------------------
_MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".webp": "image/webp", ".gif": "image/gif", ".svg": "image/svg+xml"}

_images = sorted(
    (p for p in (ROOT / "DOCUMENTS").rglob("*") if p.suffix.lower() in _MIME),
    key=lambda p: p.name,
)


def show_image(path, max_width=720):
    """Affiche une image en l'embarquant en base64 (marche aussi pour le .webp)."""
    path = Path(path)
    if not path.exists():
        return HTML(f"<p style='color:#d03b3b'>Image introuvable : {path}</p>")
    mime = _MIME.get(path.suffix.lower(), "image/png")
    b64 = base64.b64encode(path.read_bytes()).decode()
    return HTML(
        f"<img src='data:{mime};base64,{b64}' "
        f"style='max-width:{max_width}px;width:100%;border-radius:6px'>"
    )


# Par défaut : le diagramme de charge de grille
_default = next((p for p in _images if "gate_charge" in p.name), _images[0] if _images else None)

image_dd = W.Dropdown(
    options=[(p.relative_to(ROOT).as_posix(), p) for p in _images],
    value=_default,
    description="Image :",
    style={"description_width": "80px"},
    layout=W.Layout(width="620px"),
)
image_out = W.Output()


def _refresh_image(_=None):
    with image_out:
        clear_output(wait=True)
        display(show_image(image_dd.value))


image_dd.observe(_refresh_image, names="value")
_refresh_image()
display(W.VBox([image_dd, image_out]))

---
## 2. Les formules

### 2.1 Boucle de grille — d'où viennent les temps

$$R_{G,tot} = R_{drv} + R_{ext} + R_{g,int}
\qquad
I_G = \frac{V_{drive} - V_{gate}}{R_{G,tot}}
\quad\text{(borné au courant crête du driver)}$$

L'amorçage se décompose en deux sous-intervalles, et le blocage en est le miroir :

| Sous-intervalle | Charge déplacée | Tension de grille | Ce qui bouge |
|---|---|---|---|
| $t_{ri}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | le **courant** monte, $V_{ds}$ reste haute |
| $t_{fv}$ | $Q_{gd}$ | $V_{pl}$ (plateau) | la **tension** descend |
| $t_{rv}$ | $Q_{gd}$ | $V_{pl}$ | la tension remonte |
| $t_{fi}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | le courant descend |

$$t = \frac{Q_{\text{région}}}{I_G}
\qquad
t_{on} = t_{ri} + t_{fv}
\qquad
t_{off} = t_{rv} + t_{fi}$$

### 2.2 Les six pertes

$$\boxed{P_{cond} = R_{DS(on)}(T_j)\; I_{rms}^2}
\qquad
R_{DS(on)}(T_j) = R_{25}\left[1 + \alpha_R (T_j - 25)\right]$$

$$\boxed{P_{sw} = \tfrac12\left(V_{on} I_{on} t_{on} + V_{off} I_{off} t_{off}\right) f_{sw}}$$

$$\boxed{P_{oss} = \tfrac12\, C_{oss,er}(V_{on})\, V_{on}^2\, f_{sw}}
\qquad
C_{oss,er}(V) = \frac{2}{V^2}\int_0^{V} C_{oss}(v)\, v\, dv$$

$$\boxed{P_{body} = \underbrace{V_F I_{body} D_{body}}_{\text{conduction temps mort}}
+ \underbrace{Q_{rr}\, V_{on}\, f_{sw}}_{\text{recouvrement}}}$$

**Le $Q_{rr}$ se saisit à la main, et c'est celui de la diode d'en face.** La diode
que l'amorçage force à se recouvrer n'est pas celle de ce MOSFET : c'est celle du
composant opposé (Schottky parallèle, diode de roue libre discrète, autre
transistor). Ce modèle ne la connaît pas — il n'y a donc rien à en déduire
automatiquement, on donne la charge.

Sa seule conséquence de structure : $P_{rr}\propto V_{on}$, donc en **ZVS** la
valeur saisie ne coûte rien, quelle qu'elle soit.

**Pour obtenir le nombre à saisir**, la charge du point de test datasheet se
transpose au point de fonctionnement réel — $Q_{rr}$ n'est pas une propriété de la
diode seule, c'est ce que la commutation arrive à extraire :

$$Q_{rr} = Q_{rr,typ}
\left(\frac{di/dt}{(di/dt)_{ref}}\right)^{a}
\left(\frac{V_R}{V_{R,ref}}\right)^{b}
\left(\frac{I_F}{I_{F,ref}}\right)^{c}
\qquad a = b = c = \tfrac12 \text{ par défaut}$$

| Facteur | Pourquoi | Sens |
|---|---|---|
| $di/dt$ | une commutation rapide extrait la charge avant qu'elle ne se recombine | ↗ |
| $V_R$ | une tension inverse forte balaie plus fort et élargit la zone déserte | ↗ |
| $I_F$ | plus de courant direct = plus de porteurs stockés au départ | ↗ |

Le facteur en $V_R$ se recoupe bien avec la littérature : si $Q_{rr}\propto\sqrt{V_R}$
alors $E_{rr} = Q_{rr}V_R \propto V_R^{1,5}$, proche du $V^{1,4}$ couramment rapporté
pour l'énergie de recouvrement.

C'est `BODY_DIODE.q_rr_at_condition(di_dt, v_r, i_f)` — appelle-la sur la diode
concernée avec le $di/dt$ que le panneau affiche, ou lis la courbe de sa
datasheet. Les exposants $a,b,c$ sont des champs de `BODY_DIODE`, à recaler par
composant.

$$\boxed{P_{gate} = Q_g\, \Delta V_{gs}\, f_{sw}}
\qquad\text{réparti}\ \propto R:\quad
P_{g,int} = P_{gate}\frac{R_{g,int}}{R_{G,tot}}$$

**Seule la part interne chauffe la puce.** Le driver et la résistance externe dissipent
la leur hors du boîtier :

$$\boxed{P_{total} = P_{cond} + P_{sw} + P_{oss} + P_{body} + P_{g,int}}$$

### 2.3 Vitesses de commutation

$$\left.\frac{di}{dt}\right|_{on} = \frac{I_{on}}{t_{ri}}
\qquad
\left.\frac{dv}{dt}\right|_{on} = \frac{V_{on}}{t_{fv}}
\qquad
\left.\frac{dv}{dt}\right|_{off} = \frac{V_{off}}{t_{rv}}$$

### 2.4 Couplage thermique

$R_{DS(on)}$ monte avec $T_j$, $T_j$ monte avec les pertes : ni l'un ni l'autre ne se
calcule seul. Point fixe itéré jusqu'à convergence :

$$\boxed{T_j^{(k+1)} = T_{amb} + R_{th}\; P_{total}\!\left(T_j^{(k)}\right)}$$

S'il diverge, le montage est en **emballement thermique** — le notebook l'affiche.

### 2.5 Tensions commutées — la souplesse du modèle

Il n'y a **pas** de bouton « hard/soft switching » : on renseigne directement la tension
réellement balayée sur chaque front.

| Cas | $V_{turn\,on}$ | $V_{turn\,off}$ |
|---|---|---|
| Commutation dure | $V_{bus}$ | $V_{bus}$ |
| ZVS à l'amorçage | $0$ | $V_{bus}$ |
| ZVS des deux côtés | $0$ | $0$ |
| Snubbé au blocage | $V_{bus}$ | fraction de $V_{bus}$ |

Mettre $V_{turn\,on} = 0$ annule tout seul $P_{oss}$ **et** le recouvrement.

---
## 3. Panneau de calcul

Un seul champ pour le recouvrement : **le $Q_{rr}$ de la diode d'en face**, celle
que l'amorçage de ce MOSFET force à se recouvrer — Schottky parallèle, diode de
roue libre discrète, ou le transistor opposé. C'est la charge que le turn-on doit
balayer, quelle qu'en soit l'origine.

Laisser **0** quand il n'y a rien à recouvrer : ZCS, ZVS, ou GaN sans body diode.

### 3.1 Le refroidissement — un chemin par face du boîtier

Jusqu'ici le $R_{th}$ était un nombre saisi à la main. Il se calcule maintenant à partir
de la géométrie réelle : chaque face du boîtier qui évacue de la chaleur ouvre une
**branche** propre, et les branches se somment en parallèle.

```
                            ( T_j )
                               │
              ┌────────────────┴────────────────┐
        R_thJC(bottom)                    R_thJC(top)      ← datasheet, THERMAL_INFO.r_thjc
              │                                 │
        R_ext(bottom)                     R_ext(top)       ← Dissipator.get_rth()
              │                                 │
              └────────────────┬────────────────┘
                               │
                          ( T_ambiant )
```

$$\boxed{R_{thJA} = \left[\;\sum_{\text{faces actives}} \frac{1}{R_{thJC,f} + R_{ext,f}}\right]^{-1}}$$

C'est exactement ce que fait `THERMAL_INFO.r_thja_value(external_paths)` : il apparie
chaque $R_{ext}$ à son $R_{thJC}$ **par le nom de la face** (`"bottom"`, `"top"`), met les
deux en série, puis tous les résultats en parallèle.

**Deux familles de $R_{ext}$**, toutes deux dans `SRC/THERMAL/dissipator.py` :

| Classe | Ce qu'on saisit | Pour quoi |
|---|---|---|
| `StandardDissipator` | $R_{th}$ constructeur + $R_{TIM}$ | radiateur alu, pad thermique, semelle |
| `PCBDissipator` | géométrie cuivre / substrat / vias | dissipation par le PCB lui-même |

Le `PCBDissipator` met deux chemins en parallèle : convection directe sur le plan côté
pad, et traversée du substrat (FR4 nu ∥ vias thermiques) vers le plan opposé puis
convection. Sa surface utile est plafonnée par le rayon d'étalement
$r_{max} = 1{,}5\,\text{cm}\times\sqrt{t_{cu}/35\,\mu m}$ : au-delà, le cuivre en plus ne
sert à rien, la chaleur n'y arrive pas.

**Trois pièges à connaître :**

1. **Le $R_{thJA}$ datasheet n'est plus utilisé** dès qu'une face est renseignée. Il
   suppose déjà une surface de cuivre donnée (ici « FR4 6 cm², 70 µm ») : le garder en
   parallèle reviendrait à compter deux fois le même PCB.
2. **Deux faces en `PCB` comptent le même cuivre deux fois** si chacune déclare une
   surface opposée. Le PCB n'a que deux plans : soit une seule face `PCB` avec sa surface
   opposée, soit deux faces `PCB` avec surface opposée à 0. Le notebook prévient.
3. **La face `top` est presque toujours mauvaise** en boîtier classique
   ($R_{thJC,top} = 20$ °C/W ici contre 0,5 en bottom) : elle n'apporte quelque chose que
   si on y colle un vrai radiateur.


In [3]:
# --- Dissipateurs : un bloc de réglages par face du boîtier -------------------
# Les widgets sont construits ici mais affichés dans le panneau de la cellule
# suivante, pour garder tout le calcul dans une seule sortie.

_STD = {"description_width": "215px"}
_LYD = W.Layout(width="410px")


def _fd(desc, value, step=None, lo=0.0, hi=1e9):
    return W.BoundedFloatText(value=value, description=desc, style=_STD, layout=_LYD,
                              step=step, min=lo, max=hi)


# Substrats courants : (k [W/mK], épaisseur typique [mm]). "Perso" ne touche à rien.
SUBSTRATS = {
    "FR4 standard": (0.3, 1.6),
    "IMS (base alu)": (2.0, 0.1),
    "DBC Al2O3": (25.0, 0.38),
    "DBC AlN": (170.0, 0.63),
    "Perso": (None, None),
}


class FaceDissipation:
    """Réglages d'une face : aucun dissipateur, plan de cuivre PCB, ou radiateur."""

    def __init__(self, face, *, mode="none", a_cu=10.0, a_cu_other=10.0, a_pad=0.3,
                 t_cu=70.0, n_vias=24, r_th_std=5.0, h_conv=25.0):
        self.face = face

        self.mode = W.Dropdown(
            options=[("Aucun — face non refroidie", "none"),
                     ("PCB — plan de cuivre", "pcb"),
                     ("Radiateur — R_th datasheet", "std")],
            value=mode, description="Dissipation :", style=_STD, layout=_LYD,
        )

        # -- PCBDissipator
        self.substrat = W.Dropdown(options=list(SUBSTRATS), value="FR4 standard",
                                   description="Substrat :", style=_STD, layout=_LYD)
        self.a_cu = _fd("Cuivre total cette face [cm²] :", a_cu, 0.5, lo=0.01)
        self.a_cu_other = _fd("Cuivre face opposée [cm²] (0 = aucun) :", a_cu_other, 0.5)
        self.a_pad = _fd("Surface du pad thermique [cm²] :", a_pad, 0.05, lo=0.01)
        self.t_cu = _fd("Épaisseur cuivre [µm] (35 = 1 oz) :", t_cu, 35.0, lo=1.0)
        self.e_pcb = _fd("Épaisseur substrat [mm] :", 1.6, 0.1, lo=0.01)
        self.k_pcb = _fd("k substrat [W/(m·K)] :", 0.3, 0.1, lo=0.01)
        self.k_cond = _fd("k conducteur [W/(m·K)] :", 385.0, 10.0, lo=1.0)
        self.n_vias = W.BoundedIntText(value=n_vias, description="Nombre de vias :",
                                       style=_STD, layout=_LYD, min=0, max=10000)
        self.d_via = _fd("Diamètre de via [mm] :", 0.3, 0.05, lo=0.01)
        self.plating = _fd("Métallisation du via [µm] :", 25.0, 5.0, lo=0.1)
        self.filled = W.Checkbox(value=False, description="Vias bouchés (cuivre plein)",
                                 indent=False, layout=W.Layout(width="410px"))
        self.h_conv = _fd("h convection + rayonnement [W/(m²·K)] :", h_conv, 1.0, lo=0.1)

        # -- StandardDissipator
        self.r_th = _fd("R_th du radiateur [°C/W] :", r_th_std, 0.5, lo=0.001)
        self.r_tim = _fd("R_th de l'interface TIM [°C/W] :", 0.5, 0.1)

        self._box_pcb = W.VBox([self.substrat, self.a_cu, self.a_cu_other, self.a_pad,
                                self.t_cu, self.e_pcb, self.k_pcb, self.k_cond,
                                self.n_vias, self.d_via, self.plating, self.filled,
                                self.h_conv])
        self._box_std = W.VBox([self.r_th, self.r_tim])
        self.box = W.VBox([self.mode, self._box_pcb, self._box_std],
                          layout=W.Layout(margin="8px 0 0 0"))

        self.mode.observe(self._sync_mode, names="value")
        self.substrat.observe(self._sync_substrat, names="value")
        self._sync_mode()

    # -- affichage conditionnel : on ne montre que les champs qui servent
    def _sync_mode(self, _=None):
        self._box_pcb.layout.display = "" if self.mode.value == "pcb" else "none"
        self._box_std.layout.display = "" if self.mode.value == "std" else "none"

    def _sync_substrat(self, _=None):
        k, e = SUBSTRATS[self.substrat.value]
        if k is not None:
            self.k_pcb.value, self.e_pcb.value = k, e

    def build(self):
        """Instancie le Dissipator correspondant, ou None si la face est inactive."""
        if self.mode.value == "none":
            return None
        placement = Placement(self.face)
        if self.mode.value == "std":
            return StandardDissipator(
                name=f"Radiateur {self.face}", placement=placement,
                R_th=self.r_th.value, R_tim=self.r_tim.value,
            )
        return PCBDissipator(
            name=f"Plan cuivre {self.face}", placement=placement,
            A_cu_total_side_cm2=self.a_cu.value,
            A_cu_other_side_cm2=(self.a_cu_other.value or None),
            A_pad_mosfet_cm2=self.a_pad.value,
            copper_thickness_um=self.t_cu.value,
            pcb_thickness_mm=self.e_pcb.value,
            pcb_k_W_mK=self.k_pcb.value,
            k_conductor_W_mK=self.k_cond.value,
            n_vias=self.n_vias.value,
            via_diameter_mm=self.d_via.value,
            via_plating_um=self.plating.value,
            via_filled=self.filled.value,
            h_conv_eff_W_m2K=self.h_conv.value,
        )


# Une face par valeur de Placement — donc par R_thJC déclaré dans la datasheet.
# Défaut : plan de cuivre 10 cm² double face, 24 vias, léger flux d'air (h = 25).
FACES = {
    Placement.BOTTOM.value: FaceDissipation(Placement.BOTTOM.value, mode="pcb"),
    Placement.TOP.value: FaceDissipation(
        Placement.TOP.value, mode="none", a_cu=2.0, a_cu_other=0.0, n_vias=0,
        r_th_std=5.0,
    ),
}

faces_tab = W.Tab(children=[f.box for f in FACES.values()], layout=W.Layout(width="470px"))
for i, nom in enumerate(FACES):
    faces_tab.set_title(i, f"Face {nom}")


def dissipateurs():
    """Liste des dissipateurs actifs, dans l'ordre des faces."""
    return [d for d in (f.build() for f in FACES.values()) if d is not None]


def chemins_externes():
    """Format attendu par THERMAL_INFO.r_thja_value : [(R_ext, nom_de_face), ...]."""
    return [(d.get_rth(), d.placement.value) for d in dissipateurs()]

In [ ]:
# --- Panneau de commande -----------------------------------------------------
_ST = {"description_width": "185px"}
_LY = W.Layout(width="380px")


def _f(desc, value, step=None, lo=0.0, hi=1e9):
    """Champ numérique borné. lo/hi évitent les saisies aberrantes."""
    return W.BoundedFloatText(value=value, description=desc, style=_ST, layout=_LY,
                              step=step, min=lo, max=hi)


# Composants
w_mosfet = W.Dropdown(options=sorted(MOSFET_LIBRARY), description="MOSFET :",
                      style=_ST, layout=_LY)
w_driver = W.Dropdown(options=sorted(DRIVER_LIBRARY), description="Driver :",
                      style=_ST, layout=_LY)

# Point de fonctionnement
w_von = _f("V commutée à l'amorçage [V] :", 48.0, 1.0)
w_voff = _f("V commutée au blocage [V] :", 48.0, 1.0)
w_irms = _f("Courant efficace I_rms [A] :", 15.8, 0.5)
w_ion = _f("Courant commuté amorçage [A] :", 25.0, 0.5)
w_ioff = _f("Courant commuté blocage [A] :", 25.0, 0.5)
w_fsw = _f("Fréquence de découpage [kHz] :", 100.0, 10.0)

# Boucle de grille
w_rgon = _f("R grille externe amorçage [Ω] :", 2.2, 0.1)
w_rgoff = _f("R grille externe blocage [Ω] :", 1.0, 0.1)

# Body diode : conduction pendant le temps mort, rien de plus. Le recouvrement
# se saisit séparément — ce n'est pas cette diode-là qui se recouvre.
w_ibody = _f("Courant diode temps mort [A] :", 0.0, 0.5)
w_dbody = _f("Rapport cyclique diode [%] :", 0.0, 0.5, hi=100.0)

# Q_rr — celui de la diode D'EN FACE, saisi à la main. Pas de mode « auto » :
# le MOSFET seul ne connaît pas la diode que son amorçage force à se recouvrer,
# il n'y a donc rien à déduire. 0 = pas de recouvrement (ZCS, ZVS, GaN).
w_qrr = _f("Q_rr diode d'en face [nC] :", 0.0, 1.0)

# Thermique
w_tj = _f("Tj d'évaluation [°C] :", 100.0, 5.0, lo=-40.0, hi=300.0)
w_tamb = _f("Température ambiante [°C] :", 40.0, 5.0, lo=-40.0, hi=200.0)
w_th_mode = W.Dropdown(
    options=[("Dissipateurs — calculé par face", "dissip"),
             ("R_thJA datasheet", "datasheet"),
             ("R_th saisi à la main", "manual")],
    value="dissip", description="Source du R_th :", style=_ST, layout=_LY,
)
w_rth = _f("R_th jonction→ambiante [°C/W] :", 20.0, 1.0)
w_rth.disabled = True

_ETIQUETTE_TH = {"dissip": "dissipateurs", "datasheet": "datasheet", "manual": "saisi"}


def _toggle_rth(_=None):
    w_rth.disabled = w_th_mode.value != "manual"
    faces_tab.layout.display = "" if w_th_mode.value == "dissip" else "none"


w_th_mode.observe(_toggle_rth, names="value")

w_go = W.Button(description="Calculer", button_style="primary",
                icon="calculator", layout=W.Layout(width="200px", height="38px"))
out = W.Output()


def _section(titre, widgets):
    return W.VBox([W.HTML(f"<b style='font-size:13px'>{titre}</b>"), *widgets],
                  layout=W.Layout(margin="0 28px 14px 0"))


panneau = W.VBox([
    W.HBox([
        _section("Composants", [w_mosfet, w_driver]),
        _section("Point de fonctionnement", [w_von, w_voff, w_fsw]),
    ]),
    W.HBox([
        _section("Courants", [w_irms, w_ion, w_ioff]),
        _section("Boucle de grille", [w_rgon, w_rgoff]),
    ]),
    W.HBox([
        _section("Body diode / recouvrement", [w_ibody, w_dbody, w_qrr]),
        _section("Thermique", [w_tj, w_tamb, w_th_mode, w_rth]),
        _section("Refroidissement par face", [faces_tab]),
    ], layout=W.Layout(flex_flow="row wrap")),
    w_go,
    out,
])


def _operating_point():
    return OPERATING_POINT(
        v_turn_on=w_von.value,
        v_turn_off=w_voff.value,
        i_rms=w_irms.value,
        f_sw=w_fsw.value * 1e3,
        i_on=w_ion.value,
        i_off=w_ioff.value,
        r_g_ext_on=w_rgon.value,
        r_g_ext_off=w_rgoff.value,
        i_body=w_ibody.value,
        d_body=w_dbody.value / 100.0,
        # Jamais None : le repli « body diode de ce MOSFET » du modèle n'a pas
        # de sens ici, c'est la diode d'en face qui se recouvre.
        q_rr_opposite=w_qrr.value * 1e-9,
    )


def _fmt_r(valeur):
    """Formatage d'une résistance thermique, ∞ compris (via absent = chemin ouvert)."""
    return "∞" if valeur > 1e6 else f"{valeur:.1f}"


def _detail_thermique(mosfet, actifs, r_th):
    """Tableau HTML : une ligne par branche, puis le détail interne des plans PCB."""
    lignes = "".join(
        f"<tr><td style='padding:2px 18px 2px 0'>{d.name}</td>"
        f"<td>R_thJC = <b>{mosfet.thermal.r_thjc_value(d.placement.value):.2f}</b></td>"
        f"<td style='padding-left:24px'>R_ext = <b>{_fmt_r(d.get_rth())}</b></td>"
        f"<td style='padding-left:24px'>branche = "
        f"<b>{mosfet.thermal.r_thjc_value(d.placement.value) + d.get_rth():.1f} °C/W</b>"
        f"</td></tr>"
        for d in actifs
    )
    html = (
        "<b>Chemins thermiques</b>"
        f"<table style='margin-top:6px'>{lignes}"
        f"<tr><td colspan='4' style='padding-top:8px'>Mise en parallèle des branches : "
        f"<b>R_thJA = {r_th:.1f} °C/W</b> "
        f"<span style='color:#52514e'>(datasheet nu : "
        f"{mosfet.thermal.r_thja[0]:.0f} °C/W, non utilisé ici)</span></td></tr></table>"
    )

    _ORDRE = [
        ("r_spreading_max_cm", "rayon d'étalement max [cm]"),
        ("A_cu_effective_side_cm2", "cuivre utile côté pad [cm²]"),
        ("A_cu_exposed_side_cm2", "cuivre exposé à l'air [cm²]"),
        ("R_conv_side", "R convection côté pad (chemin A)"),
        ("R_pcb_thru", "R substrat nu sous le pad"),
        ("R_vias", "R vias thermiques"),
        ("R_through", "R traversée (substrat ∥ vias)"),
        ("R_conv_other", "R convection face opposée"),
        ("R_path_B", "chemin B total"),
        ("R_total", "R_ext de la face (A ∥ B)"),
    ]
    for d in actifs:
        if not isinstance(d, PCBDissipator):
            continue
        b = d.breakdown()
        corps = "".join(
            f"<tr><td style='padding:1px 18px 1px 0'>{libelle}</td>"
            f"<td><b>{_fmt_r(b[cle])}</b></td></tr>"
            for cle, libelle in _ORDRE if cle in b
        )
        html += (f"<p style='margin:10px 0 2px'><b>Détail — {d.name}</b> "
                 f"<span style='color:#52514e'>[cm, cm², °C/W]</span></p>"
                 f"<table>{corps}</table>")
    return html


def _resistance_thermique(mosfet):
    """
    (r_th, alertes, detail_html) selon la source choisie.

    r_th = None laisse loss_thermal_iteration prendre le R_thJA datasheet.
    """
    if w_th_mode.value == "manual":
        return w_rth.value, [], ""
    if w_th_mode.value == "datasheet":
        return None, [], ""

    try:
        actifs = dissipateurs()
    except Exception as err:  # géométrie refusée par pydantic
        detail = str(err).split("Value error, ")[-1].split(" [type=")[0].strip()
        return None, [("stop", f"Géométrie de dissipateur invalide — {detail}")], ""

    if not actifs:
        return None, [("warn", "Aucune face refroidie : repli sur le R_thJA datasheet "
                               f"({mosfet.thermal.r_thja[0]:.0f} °C/W). Activer au moins "
                               "une face dans « Refroidissement par face ».")], ""

    faces_dispo = [face for _, face in mosfet.thermal.r_thjc]
    manquantes = [d.placement.value for d in actifs if d.placement.value not in faces_dispo]
    if manquantes:
        return None, [("stop", f"{w_mosfet.value} n'a pas de R_thJC pour la face "
                               f"« {manquantes[0]} » (disponibles : "
                               f"{', '.join(faces_dispo)}). Compléter `r_thjc` dans "
                               "MOSFET_LIBRARY.")], ""

    alertes = []
    pcbs = [d for d in actifs if isinstance(d, PCBDissipator)]

    # Le PCB n'a que deux plans : les déclarer des deux côtés compte le même cuivre 2×.
    if len(pcbs) > 1 and all(p.A_cu_other_side_cm2 for p in pcbs):
        alertes.append(("warn", "Les deux faces sont en « PCB » et déclarent chacune un "
                                "plan opposé : le même cuivre est compté deux fois. Mettre "
                                "« Cuivre face opposée » à 0 sur l'une des deux."))
    # Cuivre au-delà du rayon d'étalement : payé en surface de carte, inutile en thermique.
    for p in pcbs:
        if p.A_cu_effective_side_cm2 < p.A_cu_total_side_cm2 - 1e-9:
            alertes.append(("warn", f"{p.name} : seuls {p.A_cu_effective_side_cm2:.1f} cm² "
                                    f"des {p.A_cu_total_side_cm2:.1f} cm² déclarés "
                                    f"participent (rayon d'étalement à "
                                    f"{p.copper_thickness_um:.0f} µm). Élargir le plan "
                                    f"n'aidera plus : cuivre plus épais, vias, ou flux d'air."))

    chemins = [(d.get_rth(), d.placement.value) for d in actifs]
    r_th = mosfet.thermal.r_thja_value(chemins)
    return r_th, alertes, _detail_thermique(mosfet, actifs, r_th)


def _controles(mosfet, op):
    """Cohérence du point de fonctionnement vis-à-vis des limites du composant."""
    alertes = []
    v_max_coss = mosfet.c_oss.vds_points[-1]
    v_sw = max(op.v_turn_on, op.v_turn_off)

    if v_sw > mosfet.v_dss_max:
        alertes.append(("stop", f"Tension commutée {v_sw:.0f} V au-dessus du V_DSS max "
                                f"du {mosfet.component_info.part_number} "
                                f"({mosfet.v_dss_max:.0f} V) — claquage."))
    if op.v_turn_on > v_max_coss:
        alertes.append(("stop", f"La courbe C_oss de la datasheet s'arrête à "
                                f"{v_max_coss:.0f} V : impossible de calculer P_oss à "
                                f"{op.v_turn_on:.0f} V sans extrapoler. Étendre "
                                f"`vds_points` / `coss_points` dans MOSFET_LIBRARY."))
    for nom, val in (("I_rms", op.i_rms), ("I amorçage", op.i_commutated_on()),
                     ("I blocage", op.i_commutated_off())):
        if val > mosfet.i_max:
            alertes.append(("warn", f"{nom} = {val:.0f} A au-dessus du calibre "
                                    f"({mosfet.i_max:.0f} A)."))
    if w_tj.value > mosfet.thermal.t_j_max:
        alertes.append(("warn", f"Tj d'évaluation {w_tj.value:.0f} °C au-dessus de "
                                f"T_j,max ({mosfet.thermal.t_j_max:.0f} °C)."))
    # P_rr = Q_rr · V_turn_on · f_sw : sous ZVS il n'y a rien pour forcer un
    # recouvrement, la charge saisie ne coûte rien.
    if w_qrr.value > 0.0 and op.v_turn_on == 0.0:
        alertes.append(("warn", f"Q_rr = {w_qrr.value:.0f} nC saisi mais V commutée à "
                                "l'amorçage nulle (ZVS) : P_rr = Q_rr · V_turn_on · f_sw "
                                "est nul de toute façon, la valeur n'a aucun effet sur "
                                "le bilan."))
    return alertes


def calculer(_=None):
    with out:
        clear_output(wait=True)
        mosfet = load_mosfet(w_mosfet.value)
        driver = load_driver(w_driver.value)
        op = _operating_point()

        r_th, alertes_th, detail_th = _resistance_thermique(mosfet)
        alertes = _controles(mosfet, op) + alertes_th
        for niveau, message in alertes:
            couleur = "#d03b3b" if niveau == "stop" else "#b26a00"
            prefixe = "Calcul impossible" if niveau == "stop" else "Attention"
            display(HTML(f"<p style='color:{couleur};margin:2px 0'>"
                         f"<b>{prefixe} :</b> {message}</p>"))
        if any(niveau == "stop" for niveau, _ in alertes):
            return

        try:
            res = loss_single_mosfet_at_temp(mosfet, driver, op, t_j=w_tj.value)
        except ValueError as err:
            display(HTML(f"<p style='color:#d03b3b'><b>Calcul impossible :</b> {err}</p>"))
            return

        th = loss_thermal_iteration(mosfet, driver, op, t_ambient=w_tamb.value, r_th=r_th)
        t_j_max = mosfet.thermal.t_j_max

        # -- synthèse
        etat = ("<span style='color:#d03b3b'><b>DIVERGE — emballement thermique</b></span>"
                if not th.converged else
                f"<span style='color:#d03b3b'><b>Tj = {th.t_j:.1f} °C > T_j,max = {t_j_max:.0f} °C</b></span>"
                if th.t_j_max_exceeded else
                f"<span style='color:#0ca30c'><b>Tj = {th.t_j:.1f} °C</b> "
                f"(marge {t_j_max - th.t_j:.0f} °C sous T_j,max)</span>")
        display(HTML(
            f"<div style='font-size:14px;line-height:1.7'>"
            f"<b>{w_mosfet.value}</b> piloté par <b>{w_driver.value}</b><br>"
            f"Pertes à Tj = {w_tj.value:.0f} °C : <b>{res.p_total:.3f} W</b> "
            f"&nbsp;·&nbsp; R_th = {th.r_th:.1f} °C/W "
            f"({_ETIQUETTE_TH[w_th_mode.value] if r_th is not None else 'datasheet'})"
            f" &nbsp;·&nbsp; {etat}</div>"
        ))

        # -- décomposition du refroidissement
        if detail_th:
            display(HTML(detail_th))

        # -- tableau (jumeau textuel des graphes : jamais de valeur cachée)
        display(HTML("<b>Bilan de pertes [W]</b>"))
        display(loss_table(res))

        # -- temps et vitesses
        display(HTML(
            "<b>Commutation</b>"
            "<table style='margin-top:6px'>"
            f"<tr><td style='padding:2px 18px 2px 0'>t_ri (montée courant)</td>"
            f"<td><b>{res.t_ri * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>di/dt amorçage</td>"
            f"<td><b>{res.di_dt_on / 1e9:.2f} A/ns</b></td></tr>"
            f"<tr><td>t_fv (descente tension)</td><td><b>{res.t_fv * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>dv/dt amorçage</td>"
            f"<td><b>{res.dv_dt_on / 1e9:.2f} V/ns</b></td></tr>"
            f"<tr><td>t_rv (montée tension)</td><td><b>{res.t_rv * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>di/dt blocage</td>"
            f"<td><b>{res.di_dt_off / 1e9:.2f} A/ns</b></td></tr>"
            f"<tr><td>t_fi (descente courant)</td><td><b>{res.t_fi * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>dv/dt blocage</td>"
            f"<td><b>{res.dv_dt_off / 1e9:.2f} V/ns</b></td></tr>"
            f"<tr><td>R_DS(on) à {w_tj.value:.0f} °C</td>"
            f"<td colspan='3'><b>{res.r_ds_on * 1e3:.3f} mΩ</b> "
            f"(soit {res.r_ds_on / mosfet.r_ds_on.r_ds_on_25:.2f} × la valeur à 25 °C)</td></tr>"
            "</table>"
        ))

        # -- le di/dt qui sert à transposer le Q_rr datasheet de la diode d'en face
        display(HTML(
            f"<p style='color:#52514e'>Q_rr saisi : <b>{w_qrr.value:.0f} nC</b> — à relever "
            f"sur la datasheet de la diode d'en face au di/dt de l'amorçage ci-dessus "
            f"({res.di_dt_on / 1e9:.2f} A/ns), via "
            f"<code>BODY_DIODE.q_rr_at_condition()</code>.</p>"
        ))

        # -- pertes hors puce
        display(HTML(
            f"<p style='color:#52514e'>Hors boîtier : {res.p_gate_drv * 1e3:.0f} mW dans le "
            f"driver, {res.p_gate_ext * 1e3:.0f} mW dans R_grille externe "
            f"(non comptés dans P_total).</p>"
        ))

        # -- graphes
        fig = plot_loss_breakdown(
            res,
            title=f"{w_mosfet.value} — bilan de pertes",
            subtitle=f"{w_von.value:.0f} V / {w_ion.value:.0f} A / {w_fsw.value:.0f} kHz "
                     f"à Tj = {w_tj.value:.0f} °C — total {res.p_total:.2f} W",
        )
        display(fig)
        plt.close(fig)

        fig = plot_thermal_iteration(
            th, t_j_max=t_j_max,
            title=f"{w_mosfet.value} — convergence de Tj",
            subtitle=f"R_th = {th.r_th:.0f} °C/W, T_amb = {w_tamb.value:.0f} °C",
        )
        display(fig)
        plt.close(fig)


w_go.on_click(calculer)
_toggle_rth()
display(panneau)
calculer()

---
## 4. Lire le résultat — et ce que le modèle ne dit pas

**Ce qui est vérifié.** L'énergie se conserve : `P_total` est exactement la somme des six
postes, les trois parts de perte grille somment à $Q_g \Delta V_{gs} f_{sw}$, et
$C_{oss,er}$ redonne bien $\int_0^V C_{oss}(v)\,v\,dv$.

**Trois limites à garder en tête :**

1. **$R_{DS(on)}(T_j)$ est linéaire, donc optimiste.** À 175 °C le modèle donne ~1,75 ×
   $R_{25}$ là où une loi en $T^{2,3}$ donne ~2,55 ×, soit **46 % d'écart**. Conséquence
   directe : la boucle thermique converge plus bas que la réalité et l'emballement est
   moins probable dans le modèle qu'en vrai. C'est le paramètre qui fausse tout
   silencieusement — à recaler sur la datasheet via `alpha_R`.

2. **Le $Q_{rr}$ vaut ce que vaut ta saisie.** Le modèle ne l'invente pas, il le
   multiplie par $V_{on} f_{sw}$ — donc toute l'incertitude est en amont. Si tu l'as
   obtenu en transposant un point de test datasheet, souviens-toi qu'à 2 A/ns on est
   typiquement 20 × au-delà du $di/dt$ de mesure, et que les exposants
   $a = b = c = \tfrac12$ sont des valeurs d'ingénieur, pas des constantes physiques.
   Quand le recouvrement pèse lourd dans le bilan, la seule vraie réponse est une
   mesure, ou une courbe $Q_{rr}$ de la datasheet lue au bon endroit.

3. **$V_{plateau}$ est pris fixe** alors qu'il dépend du courant ($V_{pl} = V_{th} +
   I_d/g_{fs}$) et de la température. Les temps de commutation en héritent.

**Points de conception à surveiller dans les résultats :**

- Un $dv/dt$ élevé au blocage (> ~5 V/ns) menace l'immunité du composant d'en face
  (réamorçage parasite par $C_{gd}$) et le mode commun à travers l'isolation.
- Si `P_sw` domine, jouer sur $R_{grille}$ ; si c'est `P_cond`, c'est le composant ou le
  refroidissement qu'il faut reprendre.
- La marge sous $T_{j,max}$ doit rester confortable : la loi $R_{DS(on)}$ étant optimiste,
  converger à 5 °C sous la limite n'est **pas** un design validé.